# Vision Interpretability Studio — Phase 1
### Model & Interpretability Core

This notebook trains a ResNet-18 **from scratch** on Imagenette, then builds
the three interpretability techniques the web app will use:

1. **Grad-CAM / Grad-CAM++** — what region of the image drove the decision
2. **Feature visualization** — what individual filters have learned to detect
3. **FGSM adversarial perturbation** — how fragile that decision is

...and exports everything to ONNX for 100% client-side inference in the browser.

**Dataset:** [Imagenette 160 px](https://www.kaggle.com/datasets/jhoward/imagenette-160-px)
by Jeremy Howard (fastai) — the canonical, creator-maintained release. 10 easily
distinguishable ImageNet classes, 70/30 train/val split, ~13k images.

**Before running:** click **Add Data** in the Kaggle notebook sidebar, search
`Imagenette 160 px`, and add the dataset by **jhoward**. Also turn on a **GPU**
accelerator under Settings (free tier — T4 x1 or P100 both work fine).


## 1. Setup

In [ ]:
import os
import json
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Data

Imagenette's Kaggle mirror unpacks to a folder named `imagenette2-160/` with
`train/` and `val/` subfolders, each containing 10 subfolders named by
ImageNet synset ID (e.g. `n01440764`). Rather than hardcode a path that might
shift between dataset versions, we search `/kaggle/input` for a `train`/`val`
pair so this cell keeps working even if Kaggle changes the mount folder name.


In [ ]:
def find_dataset_root(search_root="/kaggle/input"):
    """Locate the imagenette directory containing train/ and val/ subfolders."""
    search_root = Path(search_root)
    if not search_root.exists():
        return None
    for train_dir in search_root.rglob("train"):
        candidate_val = train_dir.parent / "val"
        if candidate_val.exists():
            return train_dir.parent
    return None


DATA_ROOT = find_dataset_root()
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find an Imagenette train/val folder under /kaggle/input. "
        "Add the 'Imagenette 160 px' dataset (by jhoward) via the Add Data panel."
    )

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
print(f"Dataset root: {DATA_ROOT}")
print(f"Train dir:    {TRAIN_DIR}")
print(f"Val dir:      {VAL_DIR}")


In [ ]:
# Imagenette's 10 synset IDs mapped to human-readable class names.
# ImageFolder sorts subfolder names alphabetically to assign class indices;
# these synset IDs already sort into the canonical Imagenette class order.
SYNSET_TO_NAME = {
    "n01440764": "tench",
    "n02102040": "English springer",
    "n02979186": "cassette player",
    "n03000684": "chain saw",
    "n03028079": "church",
    "n03394916": "French horn",
    "n03417042": "garbage truck",
    "n03425413": "gas pump",
    "n03445777": "golf ball",
    "n03888257": "parachute",
}

IMG_SIZE = 160
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# NOTE: normalization is intentionally left OUT of these transforms and
# folded into the model itself (see section 3). That keeps every tensor in
# this notebook in plain [0, 1] pixel space, which makes FGSM's epsilon
# directly interpretable as a pixel-intensity budget, and means the ONNX
# model we export later accepts raw [0, 1] pixels straight from a <canvas> —
# no normalization logic needs to be reimplemented in JavaScript.
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

eval_transforms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transforms)

CLASS_NAMES = [SYNSET_TO_NAME[c] for c in train_dataset.classes]
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Train images: {len(train_dataset)} | Val images: {len(val_dataset)}")


In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

# Sanity check: peek one batch, confirm shapes and value range.
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, label range: [{labels.min()}, {labels.max()}]")
print(f"Pixel range: [{images.min():.3f}, {images.max():.3f}]  (expect ~[0, 1])")


## 3. Model

ResNet-18, trained **from scratch** (`weights=None`) — deliberately not
fine-tuned from ImageNet-pretrained weights, so every filter this model
learns is genuinely attributable to Imagenette, which matters for an honest
interpretability story.

Normalization is folded into the model as its first op, so the model's
public input contract is "raw pixels in `[0, 1]`" end to end — same contract
the browser will use in Phase 2/3.


In [ ]:
class NormalizedResNet(nn.Module):
    """ResNet-18 with ImageNet-style normalization folded in as the first op.

    Public contract: forward() accepts a [B, 3, H, W] tensor of raw pixels in
    [0, 1] and returns [B, num_classes] logits. Every downstream tool
    (Grad-CAM, feature viz, FGSM, and eventually the browser) only ever has
    to deal with plain [0, 1] pixels.
    """

    def __init__(self, num_classes: int):
        super().__init__()
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
        self.backbone = models.resnet18(weights=None, num_classes=num_classes)

    def normalize(self, x: torch.Tensor) -> torch.Tensor:
        return (x - self.mean) / self.std

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(self.normalize(x))


model = NormalizedResNet(NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")


## 4. Training

In [ ]:
EPOCHS = 25
LR = 0.1
WEIGHT_DECAY = 5e-5
MOMENTUM = 0.9

optimizer = torch.optim.SGD(
    model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader),
)
criterion = nn.CrossEntropyLoss()


def run_epoch(loader, train: bool):
    model.train(mode=train)
    total_loss, total_correct, total_seen = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = criterion(logits, labels)

            if train:
                loss.backward()
                optimizer.step()
                scheduler.step()

            total_loss += loss.item() * images.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_seen += images.size(0)

    return total_loss / total_seen, total_correct / total_seen


In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
CHECKPOINT_PATH = "/kaggle/working/best_model.pt"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)

    marker = " *best*" if is_best else ""
    print(
        f"Epoch {epoch:2d}/{EPOCHS} | "
        f"train_loss {train_loss:.4f} train_acc {train_acc:.4f} | "
        f"val_loss {val_loss:.4f} val_acc {val_acc:.4f}{marker}"
    )

print(f"\nBest validation accuracy: {best_val_acc:.4f}")


In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=120)
plt.show()


## 5. Grad-CAM / Grad-CAM++

Both hook into `backbone.layer4` — the last convolutional block, where
activations are most semantically meaningful (earlier layers encode edges
and textures, not "objects"). Grad-CAM++ additionally weights each spatial
location by higher-order gradient terms, which tends to localize better when
an image contains multiple instances of the target class.


In [ ]:
class GradCAM:
    """Grad-CAM and Grad-CAM++ via forward/backward hooks on a target layer."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, image: torch.Tensor, class_idx: int | None = None, plus_plus: bool = False):
        """image: [1, 3, H, W] in [0, 1]. Returns (cam [H, W] in [0, 1], predicted_class_idx, confidence)."""
        self.model.zero_grad(set_to_none=True)
        logits = self.model(image)
        probs = F.softmax(logits, dim=1)

        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        confidence = float(probs[0, class_idx].item())

        logits[0, class_idx].backward()

        activations = self.activations[0]  # [C, h, w]
        gradients = self.gradients[0]      # [C, h, w]

        if plus_plus:
            grad_sq = gradients ** 2
            grad_cube = gradients ** 3
            denom = 2 * grad_sq + (activations * grad_cube).sum(dim=(1, 2), keepdim=True)
            denom = torch.where(denom != 0, denom, torch.ones_like(denom))
            alpha = grad_sq / denom
            weights = (alpha * F.relu(gradients)).sum(dim=(1, 2))
        else:
            weights = gradients.mean(dim=(1, 2))  # global average pool

        cam = F.relu((weights.view(-1, 1, 1) * activations).sum(dim=0))
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0), size=image.shape[-2:], mode="bilinear", align_corners=False,
        )[0, 0]

        cam_min, cam_max = cam.min(), cam.max()
        cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)

        return cam.cpu().numpy(), class_idx, confidence


gradcam = GradCAM(model, model.backbone.layer4)
print("Grad-CAM hooked to backbone.layer4")


In [ ]:
def overlay_heatmap(image_chw: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """image_chw: [3, H, W] in [0, 1]. cam: [H, W] in [0, 1]. Returns [H, W, 3] uint8 overlay.

    Uses the studio's cool-violet -> warm-amber scale (matching apps/web design
    tokens) instead of a default matplotlib colormap, so notebook previews and
    the eventual browser rendering look like the same product.
    """
    low = np.array([0.165, 0.141, 0.439])   # --heat-low
    high = np.array([1.0, 0.706, 0.329])    # --heat-high
    cam_rgb = low[None, None, :] + cam[:, :, None] * (high - low)[None, None, :]

    base = np.transpose(image_chw, (1, 2, 0))
    blended = (1 - alpha * cam[:, :, None]) * base + (alpha * cam[:, :, None]) * cam_rgb
    return np.clip(blended * 255, 0, 255).astype(np.uint8)


def show_gradcam_grid(n_samples: int = 6, plus_plus: bool = False, seed: int = 0):
    rng = random.Random(seed)
    indices = rng.sample(range(len(val_dataset)), n_samples)

    fig, axes = plt.subplots(2, n_samples, figsize=(2.2 * n_samples, 4.6))
    for col, idx in enumerate(indices):
        image, true_label = val_dataset[idx]
        input_tensor = image.unsqueeze(0).to(DEVICE)
        cam, pred_idx, confidence = gradcam(input_tensor, plus_plus=plus_plus)

        image_np = image.numpy()
        overlay = overlay_heatmap(image_np, cam)

        axes[0, col].imshow(np.transpose(image_np, (1, 2, 0)))
        axes[0, col].set_title(CLASS_NAMES[true_label], fontsize=9)
        axes[0, col].axis("off")

        axes[1, col].imshow(overlay)
        correct = "✓" if pred_idx == true_label else "✗"
        axes[1, col].set_title(f"{CLASS_NAMES[pred_idx]} {confidence:.0%} {correct}", fontsize=9)
        axes[1, col].axis("off")

    method = "Grad-CAM++" if plus_plus else "Grad-CAM"
    fig.suptitle(f"{method}: original (top) vs. attention overlay (bottom)")
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/{'gradcam_pp' if plus_plus else 'gradcam'}_sample.png", dpi=120)
    plt.show()


show_gradcam_grid(plus_plus=False)
show_gradcam_grid(plus_plus=True)


## 6. Feature Visualization

For a handful of filters across early/mid/late layers, synthesize the input
image that maximally activates that filter via gradient ascent on pixel
values — starting from random noise, not a real photo, so what emerges is
purely "what this filter wants to see." Mild jitter, blur, and an L2 penalty
keep the result from degenerating into high-frequency noise, a well-known
failure mode of naive activation maximization.


In [ ]:
def visualize_filter(
    layer: nn.Module,
    filter_idx: int,
    img_size: int = 160,
    steps: int = 120,
    lr: float = 0.05,
    l2_weight: float = 1e-3,
    jitter: int = 6,
    blur_every: int = 4,
) -> np.ndarray:
    """Gradient ascent on pixel values to maximize a single filter's mean activation."""
    activation_holder = {}

    def hook(module, inp, out):
        activation_holder["value"] = out

    handle = layer.register_forward_hook(hook)

    image = torch.rand(1, 3, img_size, img_size, device=DEVICE) * 0.2 + 0.4
    image.requires_grad_(True)
    optimizer = torch.optim.Adam([image], lr=lr)

    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)

        # Random jitter: shift the canvas a few pixels each step so the
        # optimizer can't overfit to one exact pixel grid (a standard trick
        # from the activation-maximization literature).
        dx, dy = random.randint(-jitter, jitter), random.randint(-jitter, jitter)
        jittered = torch.roll(image, shifts=(dx, dy), dims=(2, 3))

        model(jittered.clamp(0, 1))
        activation = activation_holder["value"][0, filter_idx]
        loss = -activation.mean() + l2_weight * (image ** 2).mean()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            image.clamp_(0, 1)
            if blur_every and step % blur_every == 0:
                # Cheap 3x3 average-pool blur, applied in-place, damps
                # high-frequency noise without a separate kornia dependency.
                blurred = F.avg_pool2d(image, kernel_size=3, stride=1, padding=1)
                image.copy_(0.85 * image + 0.15 * blurred)

    handle.remove()
    return image.detach()[0].cpu().permute(1, 2, 0).numpy()


In [ ]:
FEATURE_VIZ_TARGETS = [
    ("layer1", model.backbone.layer1, [0, 8, 16]),
    ("layer2", model.backbone.layer2, [0, 16, 32]),
    ("layer3", model.backbone.layer3, [0, 32, 64]),
    ("layer4", model.backbone.layer4, [0, 64, 128]),
]

FEATURE_VIZ_DIR = Path("/kaggle/working/artifacts/feature_viz")
FEATURE_VIZ_DIR.mkdir(parents=True, exist_ok=True)

feature_viz_metadata = []
fig, axes = plt.subplots(len(FEATURE_VIZ_TARGETS), 3, figsize=(7, 9))

for row, (layer_name, layer_module, filter_indices) in enumerate(FEATURE_VIZ_TARGETS):
    for col, filter_idx in enumerate(filter_indices):
        img = visualize_filter(layer_module, filter_idx)
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{layer_name} · filter {filter_idx}", fontsize=8)
        axes[row, col].axis("off")

        filename = f"{layer_name}_filter{filter_idx}.png"
        plt.imsave(FEATURE_VIZ_DIR / filename, img)
        feature_viz_metadata.append({
            "layer": layer_name,
            "filter_index": filter_idx,
            "file": filename,
        })

plt.tight_layout()
plt.savefig("/kaggle/working/feature_viz_gallery_preview.png", dpi=120)
plt.show()

with open(FEATURE_VIZ_DIR / "metadata.json", "w") as f:
    json.dump(feature_viz_metadata, f, indent=2)

print(f"Saved {len(feature_viz_metadata)} feature visualizations to {FEATURE_VIZ_DIR}")


## 7. Adversarial Playground (FGSM)

Fast Gradient Sign Method: nudge every pixel by `epsilon` in the direction
that most increases the loss for the *true* label. Because normalization is
folded into the model (section 3), `epsilon` here is directly a fraction of
the full `[0, 1]` pixel range — e.g. `epsilon=0.03` means "no pixel moves by
more than ~3% of full brightness," which is what makes the result look
unchanged to a human while flipping the model's prediction.


In [ ]:
def fgsm_attack(image: torch.Tensor, true_label: int, epsilon: float) -> torch.Tensor:
    """image: [1, 3, H, W] in [0, 1], requires no pre-existing grad. Returns perturbed image."""
    image = image.clone().detach().to(DEVICE).requires_grad_(True)
    label_tensor = torch.tensor([true_label], device=DEVICE)

    logits = model(image)
    loss = criterion(logits, label_tensor)
    model.zero_grad(set_to_none=True)
    loss.backward()

    perturbation = epsilon * image.grad.sign()
    adversarial = (image + perturbation).clamp(0, 1).detach()
    return adversarial


In [ ]:
def demo_adversarial(n_samples: int = 4, epsilon: float = 0.03, seed: int = 1):
    rng = random.Random(seed)
    indices = rng.sample(range(len(val_dataset)), n_samples)

    fig, axes = plt.subplots(3, n_samples, figsize=(2.4 * n_samples, 7))
    for col, idx in enumerate(indices):
        image, true_label = val_dataset[idx]
        input_tensor = image.unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            clean_logits = model(input_tensor)
        clean_probs = F.softmax(clean_logits, dim=1)[0]
        clean_pred = int(clean_probs.argmax())

        adversarial = fgsm_attack(input_tensor, true_label, epsilon)
        with torch.no_grad():
            adv_logits = model(adversarial)
        adv_probs = F.softmax(adv_logits, dim=1)[0]
        adv_pred = int(adv_probs.argmax())

        clean_np = image.numpy()
        adv_np = adversarial[0].cpu().numpy()
        diff_np = np.abs(adv_np - clean_np)
        diff_np = diff_np / (diff_np.max() + 1e-8)

        axes[0, col].imshow(np.transpose(clean_np, (1, 2, 0)))
        axes[0, col].set_title(f"{CLASS_NAMES[clean_pred]} {clean_probs[clean_pred]:.0%}", fontsize=8)
        axes[0, col].axis("off")

        axes[1, col].imshow(np.transpose(adv_np, (1, 2, 0)))
        flip = " (FLIPPED)" if adv_pred != clean_pred else ""
        axes[1, col].set_title(f"{CLASS_NAMES[adv_pred]} {adv_probs[adv_pred]:.0%}{flip}", fontsize=8)
        axes[1, col].axis("off")

        axes[2, col].imshow(np.transpose(diff_np, (1, 2, 0)))
        axes[2, col].set_title("amplified diff", fontsize=8)
        axes[2, col].axis("off")

    for ax, row_label in zip(axes[:, 0], ["clean", f"adversarial (ε={epsilon})", "perturbation"]):
        ax.set_ylabel(row_label, fontsize=9)

    plt.tight_layout()
    plt.savefig("/kaggle/working/adversarial_demo.png", dpi=120)
    plt.show()


demo_adversarial(epsilon=0.03)


## 8. Export to ONNX

Exports the full `NormalizedResNet` (normalization included) so the browser
model in Phase 2/3 only ever has to feed raw `[0, 1]` pixels. We validate
parity between PyTorch and ONNX Runtime on a real validation batch before
trusting the export — a mismatch here would otherwise surface as a
confusing bug much later, deep in the frontend.


In [ ]:
import onnx
import onnxruntime as ort

ONNX_PATH = "/kaggle/working/artifacts/model.onnx"
Path(ONNX_PATH).parent.mkdir(parents=True, exist_ok=True)

model.eval()
dummy_input = torch.rand(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    input_names=["pixel_values"],
    output_names=["logits"],
    dynamic_axes={"pixel_values": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print(f"ONNX model saved and structurally valid: {ONNX_PATH}")


In [ ]:
# Parity check: PyTorch vs. ONNX Runtime on one real validation batch.
session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

check_images, check_labels = next(iter(val_loader))
with torch.no_grad():
    torch_logits = model(check_images.to(DEVICE)).cpu().numpy()

onnx_logits = session.run(["logits"], {"pixel_values": check_images.numpy()})[0]

max_abs_diff = np.abs(torch_logits - onnx_logits).max()
print(f"Max abs difference (PyTorch vs ONNX): {max_abs_diff:.6f}")

TOLERANCE = 1e-3
assert max_abs_diff < TOLERANCE, (
    f"ONNX/PyTorch outputs diverge by {max_abs_diff:.6f}, exceeding tolerance {TOLERANCE}. "
    "Do not ship this export — investigate before proceeding to Phase 2."
)
print("Parity check passed — safe to use in the browser.")


## 9. Save Artifacts

Everything Phase 2 needs, bundled under `/kaggle/working/artifacts/` and
zipped for a single download from the Kaggle notebook's Output panel.


In [ ]:
ARTIFACTS_DIR = Path("/kaggle/working/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

with open(ARTIFACTS_DIR / "class_labels.json", "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)

with open(ARTIFACTS_DIR / "training_history.json", "w") as f:
    json.dump({
        "history": history,
        "best_val_acc": best_val_acc,
        "epochs": EPOCHS,
        "img_size": IMG_SIZE,
        "seed": SEED,
    }, f, indent=2)

# A handful of real validation images, copied in as ready-made demo samples
# for the web app (so first-time visitors have something to click without
# needing to upload their own photo).
SAMPLES_DIR = ARTIFACTS_DIR / "sample_images"
SAMPLES_DIR.mkdir(exist_ok=True)
rng = random.Random(7)
sample_indices = rng.sample(range(len(val_dataset)), 10)
for i, idx in enumerate(sample_indices):
    src_path, label = val_dataset.samples[idx]
    dest_name = f"sample_{i:02d}_{CLASS_NAMES[label].replace(' ', '_')}.jpg"
    shutil.copy(src_path, SAMPLES_DIR / dest_name)

shutil.copy("/kaggle/working/training_curves.png", ARTIFACTS_DIR / "training_curves.png")

readme_text = f"""Vision Interpretability Studio — Phase 1 artifacts
====================================================

model.onnx              Trained ResNet-18 (normalization included), opset 17.
                         Input: 'pixel_values' [B, 3, {IMG_SIZE}, {IMG_SIZE}] float32 in [0, 1].
                         Output: 'logits' [B, {NUM_CLASSES}].
class_labels.json        Ordered list of class names matching logits indices.
training_history.json    Per-epoch loss/accuracy + run metadata.
training_curves.png      Loss/accuracy plot.
feature_viz/              Precomputed filter visualizations + metadata.json.
sample_images/            10 real validation images for the app's demo gallery.

Best validation accuracy: {best_val_acc:.4f}
"""
with open(ARTIFACTS_DIR / "README.txt", "w") as f:
    f.write(readme_text)

shutil.make_archive("/kaggle/working/vision_interpretability_artifacts", "zip", ARTIFACTS_DIR)
print("Artifacts bundled: /kaggle/working/vision_interpretability_artifacts.zip")
print(readme_text)


## Next step

Download `vision_interpretability_artifacts.zip` from this notebook's
**Output** panel. In Phase 2, `model.onnx` gets placed at
`apps/web/public/models/model.onnx` and wired up via ONNX Runtime Web —
turning the Phase 0 workbench shell from a static mockup into a live,
in-browser interpretability tool.
